In [4]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt


# Prepare functions for the fit

In [1]:
#---------------------------------------------------------------------------
# Write down the likelihood
def lnlike(theta, z, zerr, R, phi):

    #------------------------------------------
    hw, phiw = theta
    #------------------------------------------
    
    z_model = hw * np.sin(phi - np.radians(phiw))
        
    sigma2 = zerr**2
    
    varb=(z - z_model)*(z - z_model)/(sigma2)
    
    return -0.5 * np.sum(varb + np.log(sigma2))
    

In [2]:
#---------------------------------------------------------------------------
# def prior
def lnprior(theta):
    
    #------------------------------------------
    hw, phiw = theta
    #------------------------------------------

    if 0.0001 < hw < 0.9 and -180. < phiw < 180.:
        return 0.0
        
    return -np.inf

In [3]:
#---------------------------------------------------------------------------
# combine likelihood and prior
def lnprob(theta, z, zerr, R, phi):
    lp = lnprior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + lnlike(theta, z, zerr, R, phi)


# Fit

In [5]:
import emcee
import corner

In [6]:
#---------------------------------------------------------------------------
# for each ring in R:

In [ ]:
#------------------------------------------------------------------------
# now make fit
#---------------------------------------------------------------------------
# now sample the above pdf using emcee
ndim = 2
nwalkers = 100

mytest = np.array([0.07, 0.1])

pos = [mytest + 1e-4*np.random.randn(ndim) for i in range(nwalkers)]

sampler = emcee.EnsembleSampler(nwalkers, ndim, lnprob, args=(z_Galcen, z_Galcen_err, R, phi_galcen))

for i, (pos, lnp, state) in enumerate(sampler.sample(pos, iterations=nsteps)):
    if (i+1) % 100 == 0:
        print("{0:.1f}%".format(100 * float(i) / nsteps))
samples = sampler.chain[:, 1000:, :].reshape((-1, ndim))

#
h0w_mcmc, phiw_mcmc = map(lambda v: (v[1], v[2]-v[1], v[1]-v[0]),
                     zip(*np.percentile(samples, [16, 50, 84],
                                        axis=0)))
print("hw: ", h0w_mcmc)
print("phiw: ", phiw_mcmc)
#fig = corner.corner(samples, labels=["h0w (kpc)","phiw (deg)","betaw (deg/kpc)","aw"])
fig = corner.corner(samples, labels=["h0w (kpc)","phiw (deg)"])
plt.savefig('/Users/eloisapoggio/Documents/Young_giants/Fit_in_rings/FitRings_emcee_triangle_R_'+this_Rmin_label+'_'+this_Rmax_label+'.png')
#
phiw_bestfit=phiw_mcmc[0]
phiw_sigmaup_bestfit=phiw_mcmc[1]
phiw_sigmadown_bestfit=phiw_mcmc[2]
#
h0w_bestfit=h0w_mcmc[0]
h0w_sigmaup_bestfit=h0w_mcmc[1]
h0w_sigmadown_bestfit=h0w_mcmc[2]